# Variables candidatas

**Pregunta:** ¿hay variables que expliquen un activo mejor que las variables de estado actuales? Se usa cuando la matriz de exposición explica poco (R² bajo) o tiene relaciones inestables.

**Método:** misma regresión semanal que `exposiciones.ipynb` (activo en euros, variables estandarizadas, Newey-West). Mide la exposición *directa* de la candidata (con las variables de estado fijas) y cuánto R² añade dentro de muestra. Para cada activo:
1. Modelo base: las variables de estado actuales.
2. Modelo base + cada candidata por separado: cuánto sube el R² y si la candidata es significativa.
3. Estabilidad: el mismo contraste antes y después de 2022.

**Criterio para incorporar una variable:** |t| > 2,58 en ambos regímenes con el mismo signo y mejora de R² ≥ 0,02 en el periodo completo, además de sentido económico. Las candidatas que son activos (índices sectoriales, Nasdaq…) no se incorporarían como variables de estado: sirven de contraste para saber con qué se mueve el activo.

**Contrastes definidos** (se elige con `CONTRASTE`):
- *cripto* (oct. 2026): liquidez neta de la Fed, M2, dólar amplio, stablecoins, Nasdaq 100. Resultado: ninguna cumple; desde 2022 se mueve más con el Nasdaq.
- *tierras raras*: yuan, bolsa china, metales industriales, mineras, baterías y litio, dólar amplio. China domina la producción y el refinado de tierras raras, y la demanda depende de vehículos eléctricos, electrónica y defensa.

**Ojo con las candidatas que son ETF sectoriales:** si comparten posiciones con el activo (LIT y PICK con REMX, por ejemplo), parte del R² es mecánico; mirar antes la composición del índice. El R² que se compara es dentro de muestra: con muchas candidatas, alguna mejora por azar (por eso el umbral de |t| y la exigencia de estabilidad).

In [1]:

import numpy as np
import pandas as pd
import requests

from nucleo.backtest import mco
from nucleo.datos import CARPETA_DATOS, fred, precios, variable
from nucleo.universo import ACTIVOS, CORTE_REGIMEN, INICIO, VARIABLES

pd.set_option("display.width", 200)

## Configuración

In [2]:
CONTRASTES = {
    "cripto": {"objetivo": ["ETH", "BTC (WBTC)"],
               "candidatas": ["Liquidez neta Fed", "M2 EE. UU.", "Stablecoins", "Nasdaq 100 (riesgo)"]},
    "tierras raras": {"objetivo": ["VanEck Rare Earth"],
                      "candidatas": ["Yuan (CNY por USD)", "Bolsa china (MCHI)", "Metales industriales (DBB)",
                                     "Mineras globales (PICK)", "Baterías y litio (LIT)"]},
}
CONTRASTE = "tierras raras"
OBJETIVO = CONTRASTES[CONTRASTE]["objetivo"]
UMBRAL_T = 2.58

# Una candidata que ya es variable de estado (p. ej. el dólar amplio, incorporado en octubre de 2026)
# duplicaría una columna de la regresión: se descarta con aviso.
from nucleo.universo import VARIABLES as _V
for _c in CONTRASTES.values():
    _dup = [x for x in _c["candidatas"] if x in _V]
    if _dup:
        print(f"Aviso: {_dup} ya son variables de estado; se quitan de las candidatas")
        _c["candidatas"] = [x for x in _c["candidatas"] if x not in _V]

## Datos

In [3]:
def semanal(s):
    return s.resample("W-FRI").last()


def stablecoins():
    """Capitalización total de stablecoins (DefiLlama, sin clave). Copia en datos/."""
    copia = CARPETA_DATOS / "stablecoins.csv"
    try:
        r = requests.get("https://stablecoins.llama.fi/stablecoincharts/all", timeout=30)
        r.raise_for_status()
        filas = [(pd.to_datetime(int(d["date"]), unit="s"), d["totalCirculatingUSD"]["peggedUSD"]) for d in r.json()]
        s = pd.Series(dict(filas)).sort_index()
        CARPETA_DATOS.mkdir(exist_ok=True)
        s.to_csv(copia)
    except Exception as e:
        print(f"Stablecoins: descarga fallida ({e}); uso la copia local")
        s = pd.read_csv(copia, index_col=0, parse_dates=True).iloc[:, 0]
    return s.rename("stablecoins")


def liquidez_neta():
    # balance de la Fed (WALCL, millones) − cuenta del Tesoro (WTREGEN, millones) − repos inversos (RRPONTSYD, miles de millones)
    walcl, tga = semanal(fred("WALCL")), semanal(fred("WTREGEN"))
    rrp = semanal(fred("RRPONTSYD")) * 1000
    return (walcl - tga - rrp.reindex(walcl.index).fillna(0)).dropna()


def yahoo(ticker):
    return semanal(precios(ticker, periodo="max"))


CARGAR = {  # nombre -> (función que devuelve el nivel semanal, tipo de cambio)
    "Liquidez neta Fed": (liquidez_neta, "log"),
    "M2 EE. UU.": (lambda: semanal(fred("M2SL").resample("D").ffill()), "log"),
    "Dólar amplio": (lambda: semanal(fred("DTWEXBGS")), "log"),
    "Stablecoins": (lambda: semanal(stablecoins()), "log"),
    "Nasdaq 100 (riesgo)": (lambda: yahoo("QQQ"), "log"),
    "Yuan (CNY por USD)": (lambda: semanal(fred("DEXCHUS")), "log"),
    "Bolsa china (MCHI)": (lambda: yahoo("MCHI"), "log"),
    "Metales industriales (DBB)": (lambda: yahoo("DBB"), "log"),
    "Mineras globales (PICK)": (lambda: yahoo("PICK"), "log"),
    "Baterías y litio (LIT)": (lambda: yahoo("LIT"), "log"),
}
niveles_cand = {}
for nombre in CONTRASTES[CONTRASTE]["candidatas"]:
    f, tipo = CARGAR[nombre]
    try:
        niveles_cand[nombre] = (f(), tipo)
    except Exception as e:
        print(f"{nombre}: sin datos ({e})")
base_niveles = {n: (semanal(variable(f, c)), t) for n, (f, c, t, en) in VARIABLES.items() if en}


def cambios(dic):
    return pd.DataFrame({n: s.diff() if t == "dif" else np.log(s).diff() for n, (s, t) in dic.items()})


def estandarizar(df):
    df = df.loc[INICIO:]
    return (df - df.mean()) / df.std()


Zb, Zc = estandarizar(cambios(base_niveles)), estandarizar(cambios(niveles_cand))
rent = pd.DataFrame({a: np.log(semanal(precios(*ACTIVOS[a], periodo="max"))).diff() for a in OBJETIVO}).loc[INICIO:]
print("Inicio de cada candidata:")
print(Zc.apply(lambda c: c.first_valid_index().date()).to_string())

/Users/iyantp_/anaconda3/envs/finanzas/lib/python3.12/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/Users/iyantp_/anaconda3/envs/finanzas/lib/python3.12/site-packages/yfinance/utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
/Users/iyantp_/anaconda3/envs/finanzas/lib/python3.12/site-packages/yfinance/utils.py:587: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace fo

Inicio de cada candidata:
Yuan (CNY por USD)            2012-01-06
Bolsa china (MCHI)            2012-01-06
Metales industriales (DBB)    2012-01-06
Mineras globales (PICK)       2012-02-10
Baterías y litio (LIT)        2012-01-06


/Users/iyantp_/anaconda3/envs/finanzas/lib/python3.12/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/Users/iyantp_/anaconda3/envs/finanzas/lib/python3.12/site-packages/yfinance/utils.py:514: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
/Users/iyantp_/anaconda3/envs/finanzas/lib/python3.12/site-packages/yfinance/utils.py:518: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={
/Use

## Contraste
Para cada activo: R² del modelo base, R² al añadir cada candidata, su coeficiente (% semanal por +1σ) y su t, en el periodo completo y en cada régimen. Cada comparación usa solo las semanas en las que existe la candidata, para que el R² base sea comparable.

In [4]:
def contraste(activo, desde=None, hasta=None):
    filas = []
    for cand in Zc.columns:
        datos = pd.concat([rent[activo], Zb, Zc[[cand]]], axis=1, sort=True).loc[desde:hasta].dropna()
        if len(datos) < 104:
            continue
        y = datos[activo]
        _, _, r2_base, n = mco(y, datos[Zb.columns])
        b, t, r2, _ = mco(y, datos[list(Zb.columns) + [cand]])
        filas.append({"candidata": cand, "semanas": n, "R² base": r2_base, "R² con candidata": r2,
                      "mejora R²": r2 - r2_base, "beta (%)": b[cand] * 100, "t": t[cand]})
    return pd.DataFrame(filas).set_index("candidata")


fmt = {"R² base": "{:.3f}", "R² con candidata": "{:.3f}", "mejora R²": "{:+.3f}", "beta (%)": "{:+.2f}", "t": "{:+.1f}"}
for activo in OBJETIVO:
    for nombre, (d, h) in {"Completo": (None, None), f"Antes de {CORTE_REGIMEN[:4]}": (None, CORTE_REGIMEN),
                           f"Desde {CORTE_REGIMEN[:4]}": (CORTE_REGIMEN, None)}.items():
        print(f"\n{activo} · {nombre}")
        display(contraste(activo, d, h).style.format(fmt)
                .apply(lambda c: ["font-weight: bold" if abs(v) > UMBRAL_T else "" for v in c], subset=["t"]))


VanEck Rare Earth · Completo


,semanas,R² base,R² con candidata,mejora R²,beta (%),t
candidata,,,,,,
Yuan (CNY por USD),773,0.339,0.339,+0.000,-0.03,-0.2
Bolsa china (MCHI),773,0.339,0.406,+0.067,+1.48,+8.7
Metales industriales (DBB),773,0.339,0.350,+0.011,+0.84,+3.8
Mineras globales (PICK),768,0.336,0.526,+0.190,+3.32,+16.9
Baterías y litio (LIT),773,0.339,0.647,+0.308,+3.35,+20.5



VanEck Rare Earth · Antes de 2022


,semanas,R² base,R² con candidata,mejora R²,beta (%),t
candidata,,,,,,
Yuan (CNY por USD),522,0.382,0.382,+0.000,-0.06,-0.4
Bolsa china (MCHI),522,0.382,0.456,+0.073,+1.57,+6.7
Metales industriales (DBB),522,0.382,0.390,+0.008,+0.77,+2.8
Mineras globales (PICK),517,0.377,0.535,+0.157,+2.79,+13.3
Baterías y litio (LIT),522,0.382,0.589,+0.207,+2.77,+13.9



VanEck Rare Earth · Desde 2022


,semanas,R² base,R² con candidata,mejora R²,beta (%),t
candidata,,,,,,
Yuan (CNY por USD),251,0.343,0.344,+0.001,+0.22,+0.9
Bolsa china (MCHI),251,0.343,0.388,+0.045,+1.26,+4.7
Metales industriales (DBB),251,0.343,0.360,+0.017,+1.04,+2.5
Mineras globales (PICK),251,0.343,0.560,+0.217,+4.32,+12.7
Baterías y litio (LIT),251,0.343,0.754,+0.412,+3.97,+16.8


## Conclusión automática
Candidatas que cumplen el criterio (|t| > 2,58 en ambos regímenes con el mismo signo y mejora de R² ≥ 0,02 en el periodo completo).

In [5]:
for activo in OBJETIVO:
    total = contraste(activo)
    a, b = contraste(activo, None, CORTE_REGIMEN), contraste(activo, CORTE_REGIMEN, None)
    validas = [c for c in total.index
               if c in a.index and c in b.index
               and abs(a.loc[c, "t"]) > UMBRAL_T and abs(b.loc[c, "t"]) > UMBRAL_T
               and np.sign(a.loc[c, "t"]) == np.sign(b.loc[c, "t"])
               and total.loc[c, "mejora R²"] >= 0.02]
    print(f"{activo}: {', '.join(validas) if validas else 'ninguna candidata cumple el criterio'}")

VanEck Rare Earth: Bolsa china (MCHI), Mineras globales (PICK), Baterías y litio (LIT)
